# Problem Statement

## Business Context

A real estate company in the Boston suburbs is actively working to gain a competitive edge by accurately forecasting the median value of homes. By utilizing historical data from the U.S. Census Bureau, they aim to improve their property valuation and market analysis strategies. The company's current valuation methods are slow and lack the precision needed to quickly identify properties that may be undervalued or overvalued in the market. The company is seeking to take the initiative to build and deploy a predictive regression model that can provide real-time, precise home value estimates based on various socioeconomic and environmental factors.

## Objective

The Data Science & Product Innovation team developed a **Boston house price prediction model** that estimates housing prices based on key features such as crime rate, average number of rooms per dwelling, property tax rate, accessibility to highways, and other neighborhood attributes. The model was initially deployed as a lightweight web application to provide real estate professionals, buyers, and policymakers with real-time, data-driven property price insights through an intuitive interface.

However, as adoption expanded across diverse user groups and real estate platforms, the **centralized deployment architecture** began to show limitations. Increased traffic led to **performance bottlenecks** and higher latency in predictions. Furthermore, attempts to replicate and share the model with distributed teams often failed due to **inconsistencies in environments, operating systems, and dependency versions**.

To overcome these challenges, the objective is to establish a **standardized, portable, and scalable deployment mechanism** that packages the house price prediction model along with its environment, dependencies, and configuration into a unified unit. This approach will:

1. Eliminate environment compatibility issues.
2. Reduce deployment errors and maintenance overhead.
3. Simplify model distribution to real estate agencies, property listing platforms, and government organizations.
4. Ensure consistent, resilient, and low-latency access to house price predictions, regardless of location or device.


# Installing and Importing the Necessary Libraries

In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 xgboost==2.1.4 joblib==1.4.2 streamlit==1.43.2 huggingface_hub==0.29.3 -q

# App Backend

## Points to note before executing the below cells
- Go to **Hugging Face**
- Open your **Profile**
- Click on **New Space**
  - Under the space creation, enter the below details
    - Space name: **Backend**
(If you were trying with different names, be cautious when using a underscore `_` in space names, such as `backend_space`, as it can cause exceptions when accessing the API URL. Always use hyphen `-` instead, like `backend-space`.)
    - Select the space SDK: **Docker**
    - Choose a Docker tempplate: **Blank**
    - Click on **Create Space**

## Flask Web Framework


In [ ]:
# Create a folder for storing the files needed for backend server deployment
import os
os.makedirs("backend_files", exist_ok=True)

In [ ]:
%%writefile backend_files/app.py
import joblib
import pandas as pd
from flask import Flask, request, jsonify

# Initialize Flask app
house_price_api = Flask("Boston House Price Predictor")

# Load the trained Boston housing model
model = joblib.load("boston_housing_model_v1_0.joblib")

# Define a route for the home page
@house_price_api.get('/')
def home():
    return "Welcome to the Boston House Price Prediction API!"

# Define an endpoint to predict price for a single house
@house_price_api.post('/v1/house')
def predict_house_price():
    # Get JSON data from the request
    house_data = request.get_json()

    # Extract relevant house features from the input data
    sample = {
        'CRIM': house_data['CRIM'],
        'ZN': house_data['ZN'],
        'INDUS': house_data['INDUS'],
        'CHAS': house_data['CHAS'],
        'NX': house_data['NX'],     # should be NOX in your dataset, check consistency
        'RM': house_data['RM'],
        'AGE': house_data['AGE'],
        'DIS': house_data['DIS'],
        'RAD': house_data['RAD'],
        'TAX': house_data['TAX'],
        'PTRATIO': house_data['PTRATIO'],
        'LSTAT': house_data['LSTAT']
    }

    # Convert the extracted data into a DataFrame
    input_data = pd.DataFrame([sample])

    # Make a prediction using the trained model
    prediction = model.predict(input_data).tolist()[0]

    # Return the prediction as a JSON response
    return jsonify({'Predicted_MEDV': prediction})

# Define an endpoint to predict price for a batch of houses
@house_price_api.post('/v1/housebatch')
def predict_house_batch():
    # Get the uploaded CSV file from the request
    file = request.files['file']

    # Read the file into a DataFrame
    input_data = pd.read_csv(file)

    # Make predictions for the batch data
    predictions = model.predict(input_data).tolist()

    # Add predictions to the DataFrame
    input_data['Predicted_MEDV'] = predictions

    # Convert results to dictionary
    result = input_data.to_dict(orient="records")

    return jsonify(result)

# Run the Flask app in debug mode
if __name__ == '__main__':
    house_price_api.run(debug=True)

Overwriting backend_files/app.py


## Dependencies File

In [ ]:
%%writefile backend_files/requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0
requests==2.28.1
uvicorn[standard]

Writing backend_files/requirements.txt


## Dockerfile

In [ ]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's working directory
COPY . .

# Install dependencies from the requirements file without using cache to reduce image size
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Define the command to start the application using Gunicorn with 4 worker processes
# - `-w 4`: Uses 4 worker processes for handling requests
# - `-b 0.0.0.0:7860`: Binds the server to port 7860 on all network interfaces
# - `app:app`: Runs the Flask app (assuming `app.py` contains the Flask instance named `app`)
CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:house_price_api"]

Writing backend_files/Dockerfile


## Uploading Files to Hugging Face Space for the Backend

**Note**: Before running the code below, ensure that the serialized ML model has been uploaded in to `backend_files` folder.

In [ ]:
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

access_key = "---------Access Keys----------"  # Your Hugging Face token created from access keys in write mode
repo_id = "----Repo Name-----/----Repo Name-------"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/backend_files",  # Local folder path
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

# App Frontend

## Points to note before executing the below cells
- Create a Streamlit space on Hugging Face by following the instructions provided on the content page titled **`Creating Spaces and Adding Secrets in Hugging Face`** from Week 1

## Streamlit for Interactive UI

In [ ]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

In [ ]:
%%writefile frontend_files/app.py
import streamlit as st
import pandas as pd
import requests

# Streamlit UI for Boston Housing Price Prediction
st.title("Boston Housing Price Prediction App")
st.write("This app predicts the median value of owner-occupied homes (`MEDV`) in $1000s based on Boston housing dataset features.")
st.write("Move the sliders below to adjust values and get a prediction.")

# Collect user input using sliders
CRIM = st.slider("Per capita crime rate by town (CRIM)", 0.0, 100.0, 0.2, 0.1)
ZN = st.slider("Proportion of residential land zoned for lots over 25,000 sq.ft. (ZN)", 0.0, 100.0, 12.0, 1.0)
INDUS = st.slider("Proportion of non-retail business acres per town (INDUS)", 0.0, 30.0, 11.0, 0.5)
NX = st.slider("Nitric oxides concentration (NX)", 0.0, 1.0, 0.55, 0.01)
RM = st.slider("Average number of rooms per dwelling (RM)", 3.0, 9.0, 6.3, 0.1)
AGE = st.slider("Proportion of owner-occupied units built prior to 1940 (AGE)", 0.0, 100.0, 65.0, 1.0)
DIS = st.slider("Weighted distances to employment centers (DIS)", 1.0, 12.0, 4.0, 0.1)
RAD = st.slider("Index of accessibility to radial highways (RAD)", 1, 24, 4, 1)
TAX = st.slider("Full-value property tax rate per $10,000 (TAX)", 100, 700, 300, 1)
PTRATIO = st.slider("Pupil-teacher ratio by town (PTRATIO)", 10.0, 25.0, 19.0, 0.1)
LSTAT = st.slider("% lower status of the population (LSTAT)", 0.0, 40.0, 12.0, 0.1)

# Categorical feature
CHAS = st.selectbox("Charles River dummy variable (CHAS)", ["0 (No)", "1 (Yes)"])
CHAS_value = 1 if CHAS.startswith("1") else 0

# Create input DataFrame
input_data = {
    'CRIM': CRIM,
    'ZN': ZN,
    'INDUS': INDUS,
    'NX': NX,
    'RM': RM,
    'AGE': AGE,
    'DIS': DIS,
    'RAD': RAD,
    'TAX': TAX,
    'PTRATIO': PTRATIO,
    'LSTAT': LSTAT,
    'CHAS': CHAS_value
}

if st.button("Predict", type='primary'):
    response = requests.post("https://<----user-name---->-<---repo name--->.hf.space/v1/house", json=input_data)    # enter user name and space name before running the cell
    if response.status_code == 200:
        result = response.json()
        predicted_price = result["Predicted_MEDV"]
        st.success(f"🏡 Predicted Median House Value: **${predicted_price * 1000:.2f}**")
    else:
        st.error("Error in API request")

# Batch Prediction
st.subheader("Batch Prediction")

file = st.file_uploader("Upload CSV file", type=["csv"])
if file is not None:
    if st.button("Predict for Batch", type='primary'):
        response = requests.post("https://<----user-name---->-<---repo name--->.hf.space/v1/housebatch", files={"file": file})    # enter user name and space name before running the cell
        if response.status_code == 200:
            result = response.json()
            st.header("Batch Prediction Results")
            st.write(result)
        else:
            st.error("Error in API request")

Writing frontend_files/app.py


## Dependencies File

In [ ]:
%%writefile frontend_files/requirements.txt
pandas==2.2.2
requests==2.28.1
streamlit==1.43.2

Writing frontend_files/requirements.txt


## Dockerfile

In [ ]:
%%writefile frontend_files/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9-slim

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

# Define the command to run the Streamlit app on port 8501 and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing frontend_files/Dockerfile


## Uploading Files to Hugging Face Space for the Frontend

In [ ]:
access_key = "---------Access Keys----------"  # Your Hugging Face token created from access keys in write mode
repo_id = "----Repo Name-----/----Repo Name-------"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/frontend_files",  # Local folder path
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

# Inferencing using Flask API


In [ ]:
import json  # To handle JSON formatting for API requests and responses
import requests  # To send HTTP requests to the deployed Flask API

import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical computations

In [ ]:
model_root_url = "https://<user_name>-<space_name>.hf.space/"  # Base URL of the deployed Flask API on Hugging Face Space; enter user name and space name before running the cell

In [ ]:
model_url = model_root_url + "/v1/house"  # Endpoint for online (single) inference

In [ ]:
model_batch_url = model_root_url + "/v1/housebatch"  # Endpoint for batch inference

## Online Inference

In [ ]:
payload = {
    "CRIM": 0.1,        # per capita crime rate
    "ZN": 18.0,         # proportion of residential land zoned
    "INDUS": 2.3,       # proportion of non-retail business acres
    "CHAS": 0,          # Charles River (0 or 1)
    "NOX": 0.55,        # nitric oxides concentration
    "RM": 6.5,          # average number of rooms
    "AGE": 65.2,        # proportion of units built before 1940
    "DIS": 4.0,         # distance to employment centers
    "RAD": 1,           # accessibility to radial highways
    "TAX": 300,         # property tax rate
    "PTRATIO": 15.3,    # pupil-teacher ratio
    "LSTAT": 12.0       # % lower status of population
}

In [ ]:
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_url, json=payload)

In [ ]:
response

<Response [200]>

In [ ]:
print(response.json())

## Batch Inference

In [ ]:
import pandas as pd

In [ ]:
boston_dataset = pd.read_csv("batch_data.csv")

In [ ]:
# List of numerical features in the dataset
numeric_features = [
    'CRIM',      # per capita crime rate by town
    'ZN',        # proportion of residential land zoned for lots > 25,000 sq.ft.
    'INDUS',     # proportion of non-retail business acres per town
    'NOX',       # nitric oxides concentration
    'RM',        # average number of rooms per dwelling
    'AGE',       # proportion of owner-occupied units built prior to 1940
    'DIS',       # weighted distances to employment centers
    'RAD',       # index of accessibility to radial highways
    'TAX',       # full-value property tax rate per $10,000
    'PTRATIO',   # pupil-teacher ratio by town
    'LSTAT'      # % lower status of the population
]

# Define predictor matrix (X) using selected numeric and categorical features
batch_input_data = boston_dataset[numeric_features]

In [ ]:
# Prepare batch input for API request
batch_input = {
    'file': batch_input_data.to_csv(header=True, index=False).encode('utf-8')
}

In [ ]:
# Send request to the model API for batch predictions
response = requests.post(
    model_batch_url,  # Model endpoint URL
    files=batch_input
)

In [ ]:
response

<Response [200]>

In [ ]:
response.text

<font size=6 color="blue">Power Ahead!</font>
___